# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |RM008_Car_ownership_by_household |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |RM008_Car_ownership_by_household.xlsx |
| Output File |car_ownership_by_household.json |
| Notebook Author | Peter |
| Last Updated |2026-07-15 |

## 1. Import

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import json

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [4]:
# File paths

input_file = "RM008_Car_ownership_by_household.xlsx"
output_file = "car_ownership_by_household.json"

raw_data_path = Path("../../data/demographic/raw/car_ownership_by_household")
output_data_path = Path("../../data/demographic/processed/car_ownership_by_household")

## 3. Load Dataset

In [5]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Car or van availability (5 categories) Code,Car or van availability (5 categories),Household composition (15 categories) Code,Household composition (15 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,-8,Does not apply,0
1,E06000001,Hartlepool,-8,Does not apply,1,One-person household: Aged 66 years and over,0
2,E06000001,Hartlepool,-8,Does not apply,2,One-person household: Other,0
3,E06000001,Hartlepool,-8,Does not apply,3,Single family household: All aged 66 years and...,0
4,E06000001,Hartlepool,-8,Does not apply,4,Single family household: Married or civil part...,0


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [6]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24825 entries, 0 to 24824
Data columns (total 7 columns):
 #   Column                                       Non-Null Count  Dtype 
---  ------                                       --------------  ----- 
 0   Lower tier local authorities Code            24825 non-null  object
 1   Lower tier local authorities                 24825 non-null  object
 2   Car or van availability (5 categories) Code  24825 non-null  int64 
 3   Car or van availability (5 categories)       24825 non-null  object
 4   Household composition (15 categories) Code   24825 non-null  int64 
 5   Household composition (15 categories)        24825 non-null  object
 6   Observation                                  24825 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 1.3+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Car or van availability (5 categories) Code',
       'Car or van availability (5 categories)',
       'Household composition (15 categories) Code',
       'Household composition (15 categories)', 'Observation'],
      dtype='object')

In [7]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Car or van availability (5 categories) Code,Car or van availability (5 categories),Household composition (15 categories) Code,Household composition (15 categories),Observation
count,24825,24825,24825.000000,24825,24825.000000,24825,24825.000000
unique,331,331,NaN,5,NaN,15,NaN
top,W06000024,Merthyr Tydfil,NaN,Does not apply,NaN,Does not apply,NaN
freq,75,75,NaN,4965,NaN,1655,NaN
mean,NaN,NaN,-0.400000,NaN,6.466667,NaN,998.320846
std,NaN,NaN,3.929456,NaN,5.487875,NaN,2023.081209
min,NaN,NaN,-8.000000,NaN,-8.000000,NaN,0.000000
25%,NaN,NaN,0.000000,NaN,3.000000,NaN,0.000000
50%,NaN,NaN,1.000000,NaN,7.000000,NaN,255.000000
75%,NaN,NaN,2.000000,NaN,11.000000,NaN,1044.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [8]:
# Cleaning code

## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [9]:
# Dataset-specific processing
selected_columns = ['Lower tier local authorities',
                    'Car or van availability (5 categories)',
                    'Household composition (15 categories)',
                    'Observation']
df_processed = df_processed[selected_columns]
df_processed = df_processed.rename(columns={
    'Car or van availability (5 categories)': 'Car Ownership',
    'Household composition (15 categories)': 'Household Category',
    'Observation': 'Car Ownership Count'
})

In [10]:
from ontology.FRAMEWORK_HOUSEHOLD_CATEGORY import FRAMEWORK_HOUSEHOLD_CATEGORY

In [11]:
df_processed = df_processed.drop(
    df_processed[df_processed["Household Category"] == "Does not apply"].index)

df_processed = df_processed.drop(
    df_processed[df_processed["Car Ownership"] == "Does not apply"].index)

In [12]:
household_mapping = {}

for new_category, old_categories in FRAMEWORK_HOUSEHOLD_CATEGORY.items():
    for old_category in old_categories:
        household_mapping[old_category] = new_category

In [13]:
df_processed["Household Category"] = (
    df_processed["Household Category"]
        .map(household_mapping)
)

In [14]:
def get_car_ownership_count_by_household_type(local_authority):
    return df_processed[
        df_processed["Lower tier local authorities"] == local_authority
    ]

In [15]:
from ontology.STUDY_AREA import STUDY_AREA

In [16]:
get_car_ownership_count_by_household_type(STUDY_AREA)

,Lower tier local authorities,Car Ownership,Household Category,Car Ownership Count
18166,Manchester,No cars or vans in household,One-person household,12424
18167,Manchester,No cars or vans in household,One-person household,29821
18168,Manchester,No cars or vans in household,Older family household: All members aged 66+,1637
18169,Manchester,No cars or vans in household,Couple household: No children,2708
18170,Manchester,No cars or vans in household,Couple household: With dependent children,3796
18171,Manchester,No cars or vans in household,Couple household: All children non-dependent,792
18172,Manchester,No cars or vans in household,Couple household: No children,5060
18173,Manchester,No cars or vans in household,Couple household: With dependent children,1632
18174,Manchester,No cars or vans in household,Couple household: All children non-dependent,221
18175,Manchester,No cars or vans in household,Lone parent household: With dependent children,9943


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [17]:
get_car_ownership_count_by_household_type("Manchester")['Car Ownership Count'].sum()

np.int64(214735)

In [18]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
Index: 18536 entries, 16 to 24824
Data columns (total 4 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Lower tier local authorities  18536 non-null  object
 1   Car Ownership                 18536 non-null  object
 2   Household Category            18536 non-null  object
 3   Car Ownership Count           18536 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 724.1+ KB


Lower tier local authorities    0
Car Ownership                   0
Household Category              0
Car Ownership Count             0
dtype: int64

## 8. Export Processed Dataset

In [19]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

# df_processed.to_json(
#     output_data_path / output_file,
#     orient="records",
#     indent=4
# )

car_ownership_by_household = get_car_ownership_count_by_household_type("Manchester")
car_ownership_by_household.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported